# Stage 1: Setup

Mount Drive, import libraries, define paths

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Mounted at /content/drive


In [2]:
BASE  = "/content/drive/MyDrive/cap-and-trade-project/"
RAW   = BASE + "data/raw/"
CLEAN = BASE + "data/clean/"
OUT   = BASE + "output/"

# Stage 2: Load cleaned data

In [3]:
converted  = pd.read_csv(CLEAN + "prices_converted.csv")
cpi_annual = pd.read_csv(CLEAN + "cpi_annual.csv", index_col="date")["cpi"]   # Series indexed by year

In [4]:
cpi_annual.index.min(), cpi_annual.loc[2007]

(np.int64(1997), np.float64(207.34416666666667))

# Stage 3: Clean EIA electricity data

In [5]:
em_raw = pd.read_csv(RAW + "emission_annual.csv")
em_raw.columns = em_raw.columns.str.strip()

em = (em_raw
      [(em_raw["Producer Type"].str.strip() == "Total Electric Power Industry")
       & (em_raw["Energy Source"].str.strip() == "All Sources")]
      .rename(columns={"Year": "year", "State": "state_abbr", "CO2\n(Metric Tons)": "co2_mt"})
      [["year", "state_abbr", "co2_mt"]]
      .query("year >= 2001 and state_abbr != 'US-TOTAL'"))
em["co2_mt"] = pd.to_numeric(
    em["co2_mt"].astype(str).str.replace(",", "", regex=False).str.strip(),
    errors="coerce")
em

,year,state_abbr,co2_mt
14267,2001,AK,4632282
14294,2001,AL,78799719
14317,2001,AR,28486171
14344,2001,AZ,47009012
14389,2001,CA,71663002
...,...,...,...
50426,2024,VT,8311
50457,2024,WA,11610306
50489,2024,WI,32334936
50507,2024,WV,43966578


In [6]:
abbr = {"AL":"Alabama","AK":"Alaska","AZ":"Arizona","AR":"Arkansas","CA":"California","CO":"Colorado",
 "CT":"Connecticut","DE":"Delaware","DC":"District of Columbia","FL":"Florida","GA":"Georgia","HI":"Hawaii",
 "ID":"Idaho","IL":"Illinois","IN":"Indiana","IA":"Iowa","KS":"Kansas","KY":"Kentucky","LA":"Louisiana",
 "ME":"Maine","MD":"Maryland","MA":"Massachusetts","MI":"Michigan","MN":"Minnesota","MS":"Mississippi",
 "MO":"Missouri","MT":"Montana","NE":"Nebraska","NV":"Nevada","NH":"New Hampshire","NJ":"New Jersey",
 "NM":"New Mexico","NY":"New York","NC":"North Carolina","ND":"North Dakota","OH":"Ohio","OK":"Oklahoma",
 "OR":"Oregon","PA":"Pennsylvania","RI":"Rhode Island","SC":"South Carolina","SD":"South Dakota",
 "TN":"Tennessee","TX":"Texas","UT":"Utah","VT":"Vermont","VA":"Virginia","WA":"Washington",
 "WV":"West Virginia","WI":"Wisconsin","WY":"Wyoming"}
em["state"] = em["state_abbr"].map(abbr)

In [7]:
# Electricity price
price_raw = pd.read_csv(RAW + "Average_retail_price_of_electricity_residential_annual.csv")

price = (price_raw
         .rename(columns=lambda c: c.replace(" cents per kilowatthour", "").strip())
         .rename(columns={"Year": "year", "District Of Columbia": "District of Columbia"})
         .melt(id_vars="year", var_name="state", value_name="res_price_cents"))
price["res_price_cents"] = pd.to_numeric(price["res_price_cents"], errors="coerce")
print(price.shape, price["state"].nunique())

# Generation
gen_raw = pd.read_csv(RAW + "annual_generation_state.csv", skiprows=0)
gen_raw.columns = gen_raw.columns.str.strip()
print(gen_raw.columns.tolist())

gen = (gen_raw
       [(gen_raw["TYPE OF PRODUCER"].str.strip() == "Total Electric Power Industry")
        & (gen_raw["ENERGY SOURCE"].str.strip() == "Total")]
       .rename(columns={"YEAR": "year", "STATE": "state_abbr",
                        "GENERATION (Megawatthours)": "generation_mwh"})
       .query("year >= 2001 and state_abbr != 'US-TOTAL'"))
gen["generation_mwh"] = pd.to_numeric(gen["generation_mwh"].astype(str).str.replace(",", ""), errors="coerce")
gen["state"] = gen["state_abbr"].map(abbr)
gen = gen[["state", "year", "generation_mwh"]]

for name, d in [("price", price), ("gen", gen), ("em", em)]:
    print(name, d.shape, d.isna().sum().to_dict())

(1275, 3) 51
['YEAR', 'STATE', 'TYPE OF PRODUCER', 'ENERGY SOURCE', 'GENERATION (Megawatthours)']
price (1275, 3) {'year': 0, 'state': 0, 'res_price_cents': 0}
gen (1289, 3) {'state': 14, 'year': 0, 'generation_mwh': 0}
em (1224, 4) {'year': 0, 'state_abbr': 0, 'co2_mt': 0, 'state': 0}


In [8]:
elec = (price.merge(gen, on=["state", "year"]).merge(em[["state", "year", "co2_mt"]], on=["state", "year"]))
elec["co2_rate"] = elec["co2_mt"] / elec["generation_mwh"]          # metric tons per MWh
elec.to_csv(CLEAN + "elec_panel.csv", index=False)

# Stage 4: Expected effect of permits

In [9]:
permit = (converted[converted["program"].isin(["California", "RGGI"])]
          .groupby(["program", "year"])["price"].mean().reset_index())       # nominal
permit.loc[permit["program"] == "RGGI", "price"] *= 1.10231              # $/short ton -> $/metric ton

rggi_core = ["Connecticut", "Delaware", "Maine", "Maryland", "Massachusetts",
             "New Hampshire", "New York", "Rhode Island", "Vermont"]
program_map = {s: "RGGI" for s in rggi_core} | {"California": "California"}
grp_elec = (elec.assign(program=elec["state"].map(program_map))
                .dropna(subset=["program"])
                .groupby(["program", "year"])
                .agg(co2=("co2_mt", "sum"), gen=("generation_mwh", "sum"), res_price=("res_price_cents", "mean"))
                .assign(co2_rate=lambda d: d["co2"] / d["gen"]).reset_index())

expected = grp_elec.merge(permit, on=["program", "year"]).query("year in [2015, 2020]")
expected["permit_cents_kwh"] = expected["price"] * expected["co2_rate"] / 10
expected["expected_share"] = expected["permit_cents_kwh"] / expected["res_price"]
expected[["program", "year", "co2_rate", "permit_cents_kwh", "res_price", "expected_share"]]

,program,year,co2_rate,permit_cents_kwh,res_price,expected_share
3,California,2015,0.282055,0.350806,16.990000,0.020648
8,California,2020,0.224997,0.383394,20.450000,0.018748
20,RGGI,2015,0.297279,0.200139,17.448889,0.011470
25,RGGI,2020,0.241955,0.170894,18.445556,0.009265


In [10]:
expected.groupby('program').mean()

,year,co2,gen,res_price,co2_rate,price,permit_cents_kwh,expected_share
program,,,,,,,,
California,2017.5,49462212.0,194893697.5,18.720000,0.253526,14.738750,0.367100,0.019698
RGGI,2017.5,75908531.0,280232691.0,17.947222,0.269617,6.897705,0.185517,0.010367


# Stage 5: Minimum detectable effect

In [11]:
# Real residential electricity price, by group
elec["real_price"] = elec["res_price_cents"] * cpi_annual.loc[2007] / elec["year"].map(cpi_annual)

group_map = ({s: "RGGI (9 core states)" for s in rggi_core}
             | {"California": "California"}
             | {s: "Excluded" for s in ["New Jersey", "Virginia", "Washington"]})
elec["group"] = elec["state"].map(group_map).fillna("Never-treated states")

print(elec["group"].value_counts())

grp_price = (elec[elec["group"] != "Excluded"]
             .groupby(["group", "year"])["real_price"].mean()
             .unstack("group"))

def mde(grp, treated_col, start, control_col="Never-treated states", end=2024):
    gap = np.log(grp[treated_col]) - np.log(grp[control_col])
    pre = gap.loc[:start - 1].dropna()
    slope, intercept = np.polyfit(pre.index, pre.values, 1)
    resid_sd = (pre - (slope * pre.index + intercept)).std(ddof=2)
    n_pre, n_post = len(pre), len(gap.loc[start:end].dropna())
    return {"program": treated_col, "resid_sd": resid_sd, "n_pre": n_pre, "n_post": n_post,
            "mde": 2.8 * resid_sd * np.sqrt(1 / n_pre + 1 / n_post)}

exp_avg = expected.groupby("program")["expected_share"].mean()
mde_elec = pd.DataFrame([mde(grp_price, "RGGI (9 core states)", 2009),
                         mde(grp_price, "California", 2013)])
mde_elec["expected"] = [exp_avg['RGGI'], exp_avg['California']]     # from the expected table (average of 2015 and 2020)
mde_elec["detectable"] = mde_elec["expected"] >= mde_elec["mde"]
mde_elec

group
Never-treated states    912
RGGI (9 core states)    216
Excluded                 72
California               24
Name: count, dtype: int64


,program,resid_sd,n_pre,n_post,mde,expected,detectable
0,RGGI (9 core states),0.034607,8,16,0.041958,0.010367,False
1,California,0.036468,12,12,0.041687,0.019698,False
